In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        if requirement is None:  # a GPU-only package on a CPU runtime
            continue
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, cpu_requirement, _, match) in pinned.items():
    if not gpu and cpu_requirement is None:
        continue
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "85bd815efd86af24338a0f0c45844cb7e1e6f155"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/85bd815efd86af24338a0f0c45844cb7e1e6f155/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Sharding the Training State

In that section, we saw that training with Adam in fp32
stores 16 bytes per parameter—four bytes each for the weight, its gradient,
and Adam's two moment estimates. For our 18-block, 1B-parameter GPT, that
fixed state fills 15.1 GiB of a 24 GiB card before reading a single token, and
data parallelism (that section, that section)
replicates all 16 bytes on every GPU. If we double the depth of our GPT to 36
blocks (1.92B parameters, 28.6 GiB of state), data parallelism on four 24 GiB
GPUs cannot train it any more than one GPU can.

Can $k$ GPUs train a model that no single card can hold? They can, because a
GPU needs the full weights of a block only while computing that block's
forward or backward pass, and it never needs the full gradient or full
optimizer state at all. Each of the $k$ GPUs can therefore store $1/k$ of the
weights, gradients, and optimizer moments, gather each block's weights on the
fly just before the block runs, and reduce-scatter each block's gradients as
soon as they are produced.

In [ ]:
# XLA reads its flags once, when JAX starts, so this cell runs before the
# imports. The latency-hiding scheduler orders the step with its memory in
# view; without it, the sharded step needs more memory than JAX may
# allocate by default (see the note on stage 2). On GPUs without
# peer-to-peer access, such as ours, set NCCL_LOCAL_REGISTER=0 in the
# environment before starting: NCCL's buffer registration fails there and
# prints a warning at every collective.
import os
os.environ['XLA_FLAGS'] = (os.environ.get('XLA_FLAGS', '') +
                           ' --xla_gpu_enable_latency_hiding_scheduler=true')

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

In [ ]:
import collections
import re
from jax.sharding import NamedSharding, PartitionSpec as P

We first verify on a meta/shape-only device that the 36-block GPT's $16N$
bytes of training state exceed the 24 GiB capacity of a single RTX 4090:

In [ ]:
config = dict(vocab_size=50257, num_hiddens=2048, num_heads=16,
              num_blks=36, pos='learned')
k = 2                       # GPUs, from the all-gather below on
key = d2l.get_key()
N = d2l.num_params(nnx.eval_shape(lambda: d2l.GPT(**config,
                                                  rngs=nnx.Rngs(key))))
print(f'{N / 1e9:.2f} billion parameters, 16N = {16 * N / 2**30:.1f} GiB')
if jax.default_backend() == 'gpu':
    limit = jax.devices()[0].memory_stats()['bytes_limit']
    print(f'JAX may allocate {limit / 2**30:.1f} GiB of one GPU')

## Partitioning the Optimizer State, Gradients, and Weights

In standard data parallelism, all $k$ GPUs hold identical copies of Adam's
moment estimates and perform identical optimizer updates. The *ZeRO*
(*Zero Redundancy Optimizer*) family [@Rajbhandari.Rasley.Ruwase.ea.2020]
eliminates this redundancy in three cumulative stages (the table):

* **ZeRO Stage 1** partitions only the optimizer state: each GPU holds Adam's
  two moments for $1/k$ of the parameters ($8 + 8/k$ bytes/parameter). Instead
  of an allreduce ($=\text{reduce-scatter} + \text{all-gather}$,
  that section), the GPUs reduce-scatter the gradients so each GPU
  receives the summed gradient for its $1/k$ shard, updates its own parameters,
  and all-gathers the updated weights—moving the exact same bytes as data
  parallelism.
* **ZeRO Stage 2** also partitions the gradients ($4 + 12/k$ bytes/parameter):
  as each gradient bucket fills during the backward pass, it is immediately
  reduce-scattered and the non-owned $1 - 1/k$ gradient shards are freed,
  again at zero extra byte cost over data parallelism.
* **ZeRO Stage 3**, or *Fully Sharded Data Parallelism* (FSDP)
  [@Zhao.Gu.Varma.ea.2023], shards the weights as well ($16/k$
  bytes/parameter). Before a block runs in the forward or backward pass, an
  all-gather assembles its full weights; immediately after the block finishes,
  the non-owned weight shards are freed, and the block's gradients are
  reduce-scattered (the figure).

<!-- tbl-caption:Training state per GPU at the end of the backward pass, with Adam in fp32 on $k$ GPUs, and for the GPT with 36 blocks and 1.92 billion parameters on $k = 2$. -->

| each GPU holds | weights | gradients | Adam state | bytes per parameter | GPT, $k = 2$ |
|---|---|---|---|---|---|
| data parallelism | all | all | all | 16 | 28.6 GiB |
| ZeRO stage 1 | all | all | $1/k$ | $8 + 8/k$ | 21.4 GiB |
| ZeRO stage 2 | all | $1/k$ | $1/k$ | $4 + 12/k$ | 17.9 GiB |
| ZeRO stage 3 (FSDP) | $1/k$ | $1/k$ | $1/k$ | $16/k$ | 14.3 GiB |

![The life of one block's weights under FSDP on one GPU. Between uses the
block is sharded, and each GPU holds $1/k$ of it; the other blocks stay
sharded throughout. An all-gather assembles the whole block for its forward
pass, after which it is released. A second all-gather assembles it for the
backward pass, after which it is released again, and a reduce-scatter
leaves each GPU with its share of the
gradients.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-fsdp-lifecycle.svg)

Because FSDP performs two all-gathers (forward and backward) and one
reduce-scatter of $S = 4N$ bytes per step—three collective phases where a ring
allreduce performs two—it moves **$1.5\times$ the communication volume of data
parallelism**. the table summarizes the four primary
collectives used across parallel training.

<!-- tbl-caption:The collective operations behind data-parallel and sharded training. -->

| collective | what every GPU ends with | where it appears |
|---|---|---|
| allreduce | the full sum | gradients in data parallelism |
| reduce-scatter | one $1/k$ shard of the sum | gradients in ZeRO and FSDP |
| all-gather | every shard, concatenated | weights in ZeRO and FSDP |
| all-to-all | a distinct slice from each GPU | token routing in mixtures of experts (that section) |

## Memory and Communication

Let us predict the peak memory and step time of our 36-block GPT under FSDP on
$k = 2$ GPUs, with 2 sequences of 1,024 tokens per GPU and per-block activation
checkpointing (that section).

* **Memory:** The sharded training state takes $16N/2 = 14.3$ GiB per GPU.
  Checkpointed activations take ~1 MiB/token (~2 GiB for 2,048 tokens), and one
  or two currently gathered blocks (~0.2 GB each) bring peak memory to at most
  **~17 GiB** per GPU—comfortably below 24 GiB.
* **Communication:** Each of the three collective phases sends $(k-1)S/k$
  bytes per GPU, giving a total communication time of

  $$
  t_\textrm{comm} = \frac{3(k-1)}{k}\,S\,\beta_\textrm{link}.
  $$

  Notice the key scaling property of FSDP: **adding more GPUs divides the
  state memory per GPU by $k$, but the bytes each GPU sends per step grow from
  $1.5S$ (at $k = 2$) toward $3S$ (at large $k$)**. Only processing more tokens
  per GPU or using a faster interconnect reduces the communication share of a
  step.

We measure $\beta_\textrm{link}$ for an all-gather of 256 MiB across two GPUs
and evaluate our paper predictions for memory, communication time, and compute
time ($8N$ FLOPs/token with checkpointing at 82.6 TFLOP/s in tf32):

In [ ]:
assert jax.device_count() >= k, 'The cells from here on need two GPUs'
mesh = jax.make_mesh((k,), ('data',), devices=jax.devices()[:k],
                     axis_types=(jax.sharding.AxisType.Auto,))
x = jnp.zeros(2**26, device=NamedSharding(mesh, P('data')))  # 256 MiB
# Asking for the result on every GPU makes the compiler gather the parts
gather = jax.jit(lambda x: x, out_shardings=NamedSharding(mesh, P()))
t = d2l.Benchmark(lambda: gather(x), repeats=10).time
beta_link = t / ((k - 1) / k * x.nbytes)  # Seconds per byte sent per GPU
del x                       # Frees its memory before the training step
print(f'bus bandwidth = {1e-9 / beta_link:.1f} GB/s in an all-gather on {k} '
      f'GPUs (beta_link = {1e9 * beta_link:.2f} ns/byte)')

In [ ]:
b, T = 2, 1024        # Sequences of T tokens per GPU and step
S = 4 * N             # Bytes of the fp32 weights, and of the gradients
peak_flops = 82.6e12  # tf32 FLOP/s of the RTX 4090
print(f'state per GPU, 16N/k: {16 * N / k / 2**30:.1f} GiB')
print(f'communication, 3(k-1)S beta_link/k: '
      f'{3 * (k - 1) / k * S * beta_link:.1f} s per step')
print(f'arithmetic at the peak, 8N bT/P: '
      f'{8 * N * b * T / peak_flops:.2f} s per step')

At ~2.2 GB/s bus bandwidth over host memory, moving $1.5S = 11.5$ GB per step
takes ~5 s, whereas the 2,048 tokens of arithmetic take only ~0.4 s. Because
0.4 s of compute cannot hide 5 s of communication, the step should take ~5 s.

## Sharded Training on Two GPUs

Now we run the sharded 36-block GPT step on two GPUs, constructing the model
so that no single GPU ever materializes the full 28.6 GiB state at once.

In JAX, `fsdp_sharding` shards each parameter tensor along its first axis
divisible by $k$ (`P('data')`), and constraining each block's activation
output to remain batch-sharded (`data`) causes XLA to emit FSDP's
all-gathers and reduce-scatters automatically:

In [ ]:
def fsdp_sharding(x):
    """Split the first axis of x whose length k divides across the GPUs;
    replicate x if there is none."""
    for i, n in enumerate(x.shape):
        if n % k == 0:
            return NamedSharding(mesh, P(*[None] * i, 'data'))
    return NamedSharding(mesh, P())

@nnx.jit
def create(key):
    # The key is an argument: XLA would fold a constant key into the program,
    # together with every weight drawn from it, as large constants
    model = d2l.GPT(**config, rngs=nnx.Rngs(key))
    opt = nnx.Optimizer(model, optax.adam(1e-4), wrt=nnx.Param)
    state = nnx.state((model, opt))
    nnx.update((model, opt), jax.lax.with_sharding_constraint(
        state, jax.tree.map(fsdp_sharding, state)))
    return model, opt

model, opt = create(d2l.get_key())  # Each GPU computes only its shards

In [ ]:
data = NamedSharding(mesh, P('data'))     # Split along the batch
block = nnx.remat(lambda blk, H: blk(H))  # Recomputed in the backward pass

def loss_fn(model, X):
    H = model.token_emb(X) + model.pos_emb(jnp.arange(X.shape[1]))
    for blk in model.blks:  # Every block's output stays split by batch
        H = jax.lax.with_sharding_constraint(block(blk, H), data)
    logits = model.token_emb.attend(model.norm(H))
    return optax.softmax_cross_entropy_with_integer_labels(logits, X).mean()

# Donated, the old weights and moment estimates make room for the new ones
@nnx.jit(donate_argnums=(0, 1))
def train_step(model, opt, X):
    loss, grads = nnx.value_and_grad(loss_fn)(model, X)
    opt.update(model, grads)
    return loss

In [ ]:
X = jax.device_put(jax.random.randint(d2l.get_key(), (k * b, T), 0,
                                      config['vocab_size']), data)
compiled = train_step.lower(model, opt, X).compile()
mem = compiled.memory_analysis()  # Bytes per GPU
print(f'per GPU: weights and moments '
      f'{mem.argument_size_in_bytes / 2**30:.1f} GiB (donated: '
      f'{mem.alias_size_in_bytes / 2**30:.1f} GiB), temporaries '
      f'{mem.temp_size_in_bytes / 2**30:.1f} GiB')

In [ ]:
def collective_megabytes(compiled):
    """Megabytes of the results of each kind of collective in a compiled
    program, on one GPU."""
    sizes = collections.Counter()
    for line in compiled.as_text().splitlines():
        # A synchronous collective, or the -done half of an asynchronous
        # pair, whose result type lists only the arrays that it returns
        op = re.search(r' = (.*?) (all-gather|reduce-scatter|all-reduce|'
                       r'all-to-all|collective-permute)(-done)?\(', line)
        for bits, dims in re.findall(r'[a-z]+(\d+)\[([\d,]*)\]',
                                     op[1] if op else ''):
            n = math.prod(int(d) for d in dims.split(',') if d)
            sizes[op[2]] += int(bits) // 8 * n / 1e6
    return dict(sizes)

In [ ]:
for kind, mb in sorted(collective_megabytes(compiled).items(),
                       key=lambda kv: -kv[1]):
    print(f'{kind}: {mb / 1e3:.2f} GB')

In [ ]:
train_step(model, opt, X).block_until_ready()  # Warm-up, finished
times = [d2l.Benchmark(lambda: train_step(model, opt, X), warmup=0,
                       repeats=1).time for _ in range(3)]
kernels = d2l.kernel_times(lambda: train_step(model, opt, X), steps=1)
comm = sum(ms for name, ms in kernels.items() if 'nccl' in name.lower()) / 1e3
peak_bytes = jax.devices()[0].memory_stats()['peak_bytes_in_use']

In [ ]:
print(f'per GPU: peak memory {peak_bytes / 2**30:.1f} GiB; '
      f'{np.median(times):.2f} s per step ({min(times):.2f} to '
      f'{max(times):.2f})\nof each step, GPU 0 spends {comm:.2f} s in '
      f'communication kernels')

Both GPUs hold the 36-block model at a peak of ~16 GiB per card (under our 17
GiB bound), and each step takes ~5.5 s, with GPU 0 spending over 90% of the
step in NCCL communication kernels—within a sixth of our 5 s paper prediction.
Over NVLink 5 (900 GB/s per direction), the same 11.5 GB of collectives takes
only ~13 ms, less than a thirtieth of the 0.4 s compute time.

This comparison also shows *when* to shard: a model whose training state fits
on a single GPU alongside a reasonable micro-batch (such as our 18-block GPT)
trains much faster with data parallelism and gradient accumulation (which
communicates once per global step) than with FSDP over PCIe (which re-gathers
the weights twice on every micro-batch).

## Summary

Data parallelism replicates all $16N$ bytes of fp32 Adam state on every GPU.
ZeRO partitions this state across $k$ GPUs: Stage 1 shards the optimizer
moments ($8 + 8/k$ bytes/parameter) and Stage 2 also shards the gradients
($4 + 12/k$ bytes/parameter), both moving the exact same $2(k-1)S/k$ bytes per
step as a ring allreduce. Stage 3 (FSDP) shards the weights as well ($16/k$
bytes/parameter), gathering each block's weights for the forward and backward
pass and reduce-scattering gradients at $1.5\times$ the communication volume
of data parallelism ($3(k-1)S/k$ bytes per GPU). Sharding enables two 24 GiB
GPUs to train a 36-block GPT whose 28.6 GiB state fits on neither card alone.

## Exercises

1. **Cost of sharding.** ● Count communication per GPU and step in units
   of $S = 4N$, the bytes of one full set of fp32 weights or gradients.
    1. A variant of ZeRO Stage 3 keeps every gathered block in memory from its
       forward pass until its backward pass finishes. Which collective does
       this eliminate, and how much memory saving from the table
       does it give up?
    1. Mixed-precision FSDP all-gathers weights in bf16 and reduce-scatters gradients in bf16 while keeping each GPU's master weight and Adam shards in fp32. Compare its bytes transferred per step with fp32 data parallelism, and its memory per parameter with the last row of the table.
    1. For the 36-block GPT on $k = 2$ GPUs at the measured
       $\beta_\textrm{link}$, how many tokens per GPU would a step need for
       checkpointed compute ($8N$ FLOPs/token at 83 TFLOP/s) to match the
       communication time the equation? Do those activations fit on
       the card, and why does gradient accumulation not help when FSDP
       re-gathers weights on every micro-batch?

2. [extended] **Sharded optimizer state.** ● Implement ZeRO Stage 1
    [@Rajbhandari.Rasley.Ruwase.ea.2020] declaratively starting from
    `train_step` (that section) with `optax.adam`. Keep the
    model parameters replicated (`P()`), and place the leaves of
    `nnx.state(opt)` with `fsdp_sharding`.
    1. Ensure parameters remain replicated at the end of `train_step` via
       `jax.sharding.reshard` if needed, and verify that multi-step parameter
       updates match replicated Adam.
    1. Inspect the compiled step with `collective_megabytes` and compare its collectives and live/peak memory at $k = 2$ and $k = 4$ against the table.

3. [code] **Tensor versus data parallelism.** ● Consider the MLP block
    $\mathbf{Y} = \phi(\mathbf{X}\mathbf{W}_1)\mathbf{W}_2$ with
    $\mathbf{X} \in \mathbb{R}^{n \times d}$,
    $\mathbf{W}_1 \in \mathbb{R}^{d \times 4d}$, and
    $\mathbf{W}_2 \in \mathbb{R}^{4d \times d}$ across $k$ devices. Tensor
    parallelism (the figure, [@shoeybi2019megatron]) splits
    $\mathbf{W}_1$ column-wise (`P(None, 'data')`) and $\mathbf{W}_2$ row-wise
    (`P('data', None)`).
    1. Show that each device can compute a partial $n \times d$ output without
       communication and that summing the $k$ partial outputs gives
       $\mathbf{Y}$. Which collective does the forward pass need, and which
       does the backward pass need for $\nabla_{\mathbf{X}}$?
    1. Compare the values sent per device and step under tensor parallelism
       versus data parallelism, and find the token count $n$ below which
       tensor parallelism sends less data. Why is tensor parallelism
       nevertheless restricted to fast intra-node NVLink domains?
    1. Place $\mathbf{W}_1$ with `P(None, 'data')`, $\mathbf{W}_2$ with
       `P('data', None)`, and $\mathbf{X}$ replicated on an `Auto` mesh, jit
       the forward and backward pass, and inspect the emitted collectives.

4. **Prefetching the next block.** In the forward pass under FSDP, let $g$ be
   the all-gather time of one block and $c$ its compute time. Without overlap,
   $L$ blocks take $L(g + c)$; prefetching issues the all-gather of block
   $l + 1$ while block $l$ computes.
    1. Show that prefetching reduces the forward pass time to
       $g + (L-1)\max(g, c) + c$, and explain why no schedule can do better
       than $L\max(g, c)$.
    1. Evaluate $g$ and $c$ for one block of the 36-block GPT ($50$M
       parameters, 2,048 tokens/GPU, 83 TFLOP/s) on our PCIe link versus
       NVLink 5 (900 GB/s), and state how much prefetching saves on each
       interconnect.

[Discussions](https://d2l.discourse.group/)

[Discussions](https://d2l.discourse.group/t/36886)